# Emergen-AI: ED Triage Recommendation\n\nThis notebook reproduces data preparation, EDA, model training, and evaluation for the ED triage recommendation system.

## Setup\nDefine paths and imports.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
REPORTS_DIR = ROOT / 'reports'
EDA_DIR = REPORTS_DIR / 'eda'
METRICS_PATH = REPORTS_DIR / 'triage_metrics.json'


## Data Acquisition & Preparation\nThe pipeline expects MIMIC-IV ED tables in `data/raw/`. Enable the flag to run preprocessing.

In [ ]:
import subprocess

RUN_PREPROCESS = False
if RUN_PREPROCESS:
    subprocess.run(['python', 'src/data_preprocessing.py'], cwd=ROOT, check=True)
    subprocess.run(['python', 'src/feature_engineering.py'], cwd=ROOT, check=True)


## Exploratory Data Analysis (EDA)\nGenerate EDA plots and view a few key visuals.

In [ ]:
RUN_EDA = False
if RUN_EDA:
    subprocess.run(['python', 'src/eda_visualization.py'], cwd=ROOT, check=True)

# Display a few saved plots if they exist
for name in ['count_acuity.png', 'count_arrival_hour.png', 'corr_heatmap.png']:
    path = EDA_DIR / name
    if path.exists():
        display(Image(filename=str(path)))


## Model Development\nTrain baseline, context-aware, graph-based, and pairwise ranking models.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run(['python', 'src/triage_pipeline.py'], cwd=ROOT, check=True)


## Evaluation\nLoad metrics and compare accuracy and relevance.

In [ ]:
if METRICS_PATH.exists():
    metrics = json.loads(METRICS_PATH.read_text())
    rows = []
    for model in ['baseline', 'context', 'graph', 'pairwise']:
        d = metrics[model]
        rows.append({
            'Model': model,
            'Accuracy': d.get('accuracy'),
            'ROC-AUC': d.get('roc_auc'),
            'PR-AUC': d.get('pr_auc'),
            'F1': d.get('f1'),
            'F2': d.get('f2'),
            'Precision@5': d.get('k=5', {}).get('precision@k'),
            'Recall@5': d.get('k=5', {}).get('recall@k'),
            'Precision@10': d.get('k=10', {}).get('precision@k'),
            'Recall@10': d.get('k=10', {}).get('recall@k'),
        })
    df = pd.DataFrame(rows)
    display(df)
